In [9]:
from torch import nn
import numpy as np
import matplotlib as mlp
import pandas as pd
import h5py
from time import sleep
from torch.utils.data import Dataset, DataLoader, random_split, Subset

In [ ]:
from model import H5Data
path = "./data.hdf5"
data = H5Data(path)

In [5]:
from sklearn.model_selection import train_test_split

labels = [int(float(group)) for group, _ in data.index_map]
seed = 1
train_idx, test_idx = train_test_split(
    range(len(data.index_map)),
    train_size=0.8,
    stratify=labels,
    random_state=seed
)
train_sub = Subset(data, train_idx)
test_sub = Subset(data, test_idx)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_sub, batch_size=BATCH_SIZE)
print(f"Length of train dataloader: {len(train_dataloader)}")
print(f"Length of test dataloader: {len(test_dataloader)}")

Length of train dataloader: 81261
Length of test dataloader: 20316


In [6]:
import torch
from model import star_tracker_v1

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = star_tracker_v1(25, 9029)
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

In [ ]:
epochs = 100
for epoch in range(epochs):
    print(f"Epoch: {epoch}")

    train_loss, train_accuracy = 0, 0
    model.train()
    for batch, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_pred = torch.argmax(y_pred, dim=1)
        train_accuracy += (train_pred == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader) * BATCH_SIZE

    test_loss, test_accuracy = 0, 0
    model.eval()
    with torch.inference_mode():
        for x,y in (test_dataloader):
            x, y = x.to(device), y.to(device)

            y_pred = model(x)
            test_loss += loss_fn(y_pred, y).item()
            test_pred = torch.argmax(y_pred, dim=1)
            test_accuracy += (test_pred == y).sum().item()

        test_loss /= len(test_dataloader)
        test_accuracy /= len(test_dataloader) * BATCH_SIZE

    print(f"\nTrain loss: {train_loss:.5f} | Train Accuracy: {train_accuracy:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_accuracy:.2f}\n")

Epoch: 0
